In [ ]:
%pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
%pip install -U "transformers>=5.6.0" "accelerate>=1.13.0" "datasets>=4.8.0" "trl>=0.23.0" "peft>=0.19.0" "bitsandbytes>=0.49.0" "sentencepiece>=0.2.0" "protobuf>=5.28.0" "huggingface_hub>=0.34.0" "tensorboard>=2.20.0"


In [ ]:
import torch
import bitsandbytes as bnb

print("torch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
print("bitsandbytes:", bnb.__version__)

if not torch.cuda.is_available():
    raise RuntimeError("Pytorch has no CUDA support. Please check your installation.")


In [ ]:
import gc
from contextlib import nullcontext
import json
import math
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from datasets import Dataset, DatasetDict
from huggingface_hub import login
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, EarlyStoppingCallback, TrainerCallback, pipeline, set_seed
from trl import SFTConfig, SFTTrainer

torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
set_seed(42)

print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"gpu: {props.name}")
    print(f"compute capability: {torch.cuda.get_device_capability(0)}")
    print(f"vram: {props.total_memory / 1024**3:.2f} GB")


In [ ]:
DATA_PATH = Path("training_data_LLM_format")
OUTPUT_DIR = Path("gemma4_legal_process_lora") / "e2b_it_gdpr_qlora_adapter_one_shot"
MERGED_OUTPUT_DIR = Path("gemma4_legal_process_lora") / "e2b_it_gdpr_qlora_merged_one_shot"

BASE_MODEL_ID = "google/gemma-4-E2B-it"
TOKENIZER_ID = "google/gemma-4-E2B-it"

SEED = 42
TRAIN_SPLIT = 0.85
VALIDATION_SPLIT = 0.10
TEST_SPLIT = 0.05
MAX_SEQ_LENGTH = "auto_fit_dataset"
AUTO_MAX_SEQ_HEADROOM = 512
AUTO_MAX_SEQ_ROUND_TO = 1024
AUTO_MAX_SEQ_HARD_CAP = 8192
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

NUM_TRAIN_EPOCHS = 10
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 0.2
EARLY_STOPPING_PATIENCE = 6
TRAIN_EXAMPLE_PROGRESS_EVERY = 1
LOGGING_STEPS = 1
EVAL_SAVE_STEPS = 10
SAVE_TOTAL_LIMIT = 3
INFERENCE_MAX_NEW_TOKENS = 8192

DEBUG_VERBOSE_TRAINING = True
DEBUG_VERBOSE_EVAL = True
DEBUG_SHOW_GPU_MEMORY = True
DEBUG_SYNCHRONIZE_FOR_TIMINGS = True
DEBUG_PEEK_BATCHES = 3
DEBUG_RUN_BATCH_TIMING_PROBE = True

MAX_ALLOWED_ALLOCATED_BEFORE_MODEL_LOAD_GB = 1.0
REQUIRE_GPU_ONLY_MODEL_PLACEMENT = True
MODEL_DEVICE_MAP = {"": 0}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_PATH = {DATA_PATH.resolve()}")
print(f"OUTPUT_DIR = {OUTPUT_DIR.resolve()}")
print(f"BASE_MODEL_ID = {BASE_MODEL_ID}")
print(f"TOKENIZER_ID = {TOKENIZER_ID}")


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(tokenizer.special_tokens_map)
print("Chat template:", tokenizer.chat_template is not None)


In [ ]:
def load_jsonl(path: Path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Wrong JSON format in line {line_number}: {exc}") from exc
    return records


def validate_record(record, line_number):
    for field in ["article_id", "prompt", "completion", "full_text"]:
        value = record.get(field)
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f"Field '{field}' is missing or empty in line {line_number}.")

    if record["full_text"] != record["prompt"] + record["completion"]:
        raise ValueError(f"prompt + completion does not match full_text in line {line_number}.")

    source_messages = record.get("source_messages")
    if source_messages is not None:
        if not isinstance(source_messages, list) or len(source_messages) < 3:
            raise ValueError(f"source_messages is incomplete in line {line_number}.")
        roles = [message.get("role") for message in source_messages]
        if roles[:2] != ["system", "user"] or roles[-1] != "assistant":
            raise ValueError(f"Unexpected role sequence in source_messages, line {line_number}: {roles}")

    if "full_text_token_count" not in record:
        record["full_text_token_count"] = len(tokenizer(record["full_text"], add_special_tokens=False)["input_ids"])

    return record


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Data set not found: {DATA_PATH}. Please run util/convert_gdpr_process_format_to_json.py first."
    )

raw_records = load_jsonl(DATA_PATH)
records = [validate_record(record, idx) for idx, record in enumerate(raw_records, start=1)]

print(f"Anzahl Beispiele: {len(records)}")
print(f"Erstes article_id: {records[0]['article_id']}")
print("Prompt-Vorschau:\n")
print(records[0]["prompt"][:1500])
print("\nCompletion-Vorschau:\n")
print(records[0]["completion"][:1200])


In [ ]:
if not math.isclose(TRAIN_SPLIT + VALIDATION_SPLIT + TEST_SPLIT, 1.0, rel_tol=0.0, abs_tol=1e-9):
    raise ValueError("TRAIN_SPLIT + VALIDATION_SPLIT + TEST_SPLIT muss 1.0 ergeben.")

holdout_split = VALIDATION_SPLIT + TEST_SPLIT
test_share_of_holdout = TEST_SPLIT / holdout_split

dataset = Dataset.from_list(records).shuffle(seed=SEED)
train_holdout = dataset.train_test_split(test_size=holdout_split, seed=SEED)
validation_test = train_holdout["test"].train_test_split(test_size=test_share_of_holdout, seed=SEED)

dataset = DatasetDict(
    {
        "train": train_holdout["train"],
        "validation": validation_test["train"],
        "test": validation_test["test"],
    }
)

print(dataset)
print({split: len(dataset[split]) for split in dataset})
print(f"Split-Ziel: train={TRAIN_SPLIT:.0%}, validation={VALIDATION_SPLIT:.0%}, test={TEST_SPLIT:.0%}")


In [ ]:
split_rows = []
all_lengths = []

for split_name, split_dataset in dataset.items():
    split_lengths = [int(row["full_text_token_count"]) for row in split_dataset]
    all_lengths.extend(split_lengths)
    split_array = np.array(split_lengths)
    split_rows.append(
        {
            "split": split_name,
            "count": int(split_array.size),
            "min": int(split_array.min()),
            "median": int(np.median(split_array)),
            "p90": int(np.quantile(split_array, 0.90)),
            "p95": int(np.quantile(split_array, 0.95)),
            "max": int(split_array.max()),
        }
    )

length_df = pd.DataFrame(split_rows)
display(length_df)

all_lengths = np.array(all_lengths)
dataset_max_length = int(all_lengths.max())

if MAX_SEQ_LENGTH == "auto_fit_dataset":
    resolved_max_seq_length = int(
        math.ceil((dataset_max_length + AUTO_MAX_SEQ_HEADROOM) / AUTO_MAX_SEQ_ROUND_TO) * AUTO_MAX_SEQ_ROUND_TO
    )
elif isinstance(MAX_SEQ_LENGTH, str):
    raise ValueError(f"Unbekannte MAX_SEQ_LENGTH-Einstellung: {MAX_SEQ_LENGTH}")
else:
    resolved_max_seq_length = int(MAX_SEQ_LENGTH)

if AUTO_MAX_SEQ_HARD_CAP is not None:
    resolved_max_seq_length = min(resolved_max_seq_length, int(AUTO_MAX_SEQ_HARD_CAP))

truncation_share = float((all_lengths > resolved_max_seq_length).mean() * 100)
effective_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
updates_per_epoch = math.ceil(len(dataset["train"]) / effective_batch_size)
estimated_total_updates = updates_per_epoch * NUM_TRAIN_EPOCHS

print(f"Longest Example: {dataset_max_length}")
print(f"Used MAX_SEQ_LENGTH: {resolved_max_seq_length}")
print(f"Estimated truncation share: {truncation_share:.2f}%")
print(f"Effective batch size: {effective_batch_size}")
print(f"Estimated optimizer updates per epoch: {updates_per_epoch}")
print(f"Estimated total optimizer updates: {estimated_total_updates}")


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Dieses Notebook braucht fuer QLoRA eine CUDA-faehige NVIDIA-GPU.")

for stale_name in ["model", "trainer", "base_model", "peft_model", "merged_model", "inference_model", "pipe"]:
    if stale_name in globals():
        print(f"Entferne altes Objekt aus globals(): {stale_name}", flush=True)
        del globals()[stale_name]

ip = None
try:
    ip = get_ipython()
except NameError:
    ip = None

if ip is not None:
    for history_name in ["_", "__", "___"]:
        if history_name in ip.user_ns:
            print(f"Entferne Jupyter-History-Referenz: {history_name}", flush=True)
            del ip.user_ns[history_name]
    out_store = ip.user_ns.get("Out")
    if isinstance(out_store, dict) and out_store:
        print(f"Leere Jupyter Out-History mit {len(out_store)} Eintraegen", flush=True)
        out_store.clear()

gc.collect()
torch.cuda.empty_cache()
try:
    torch.cuda.ipc_collect()
except Exception:
    pass
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

major, _ = torch.cuda.get_device_capability(0)
compute_dtype = torch.bfloat16 if major >= 8 else torch.float16
allocated_before = torch.cuda.memory_allocated() / 1024**3
reserved_before = torch.cuda.memory_reserved() / 1024**3
print(f"compute_dtype = {compute_dtype}", flush=True)
print(f"Before Model Load: allocated={allocated_before:.2f} GB | reserved={reserved_before:.2f} GB", flush=True)

if allocated_before > MAX_ALLOWED_ALLOCATED_BEFORE_MODEL_LOAD_GB:
    raise RuntimeError(
        f"Before the model load, {allocated_before:.2f} GB of GPU memory is already allocated. "
        "This indicates that old model objects or another active Python/Jupyter process is holding onto GPU memory. "
        "Please restart the kernel and check with nvidia-smi if any other Python processes are holding GPU memory."
    )

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_storage=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map=MODEL_DEVICE_MAP,
    torch_dtype=compute_dtype,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
    quantization_config=bnb_config,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

allocated_after = torch.cuda.memory_allocated() / 1024**3
reserved_after = torch.cuda.memory_reserved() / 1024**3
peak_after = torch.cuda.max_memory_allocated() / 1024**3
print(type(model).__name__)
print(model.config.model_type)
print(f"Geladenes Modell laut BASE_MODEL_ID: {BASE_MODEL_ID}", flush=True)
print(f"config._name_or_path: {getattr(model.config, '_name_or_path', 'n/a')}", flush=True)
print(f"architectures: {getattr(model.config, 'architectures', None)}", flush=True)
text_config = getattr(model.config, 'text_config', model.config)
for attr in ['num_hidden_layers', 'hidden_size', 'intermediate_size', 'num_attention_heads', 'num_key_value_heads', 'vocab_size']:
    print(f"text_config.{attr} = {getattr(text_config, attr, 'n/a')}", flush=True)
print(f"Nach Modell-Load: allocated={allocated_after:.2f} GB | reserved={reserved_after:.2f} GB | peak={peak_after:.2f} GB", flush=True)

device_map = getattr(model, "hf_device_map", None)
if device_map:
    placement_counts = {}
    for placement in device_map.values():
        placement_counts[str(placement)] = placement_counts.get(str(placement), 0) + 1
    print(f"hf_device_map placements: {placement_counts}", flush=True)
    has_cpu_or_disk_offload = any(str(placement) in {"cpu", "disk"} for placement in device_map.values())
    if has_cpu_or_disk_offload and REQUIRE_GPU_ONLY_MODEL_PLACEMENT:
        raise RuntimeError(
            "The model is partially offloaded to CPU or disk according to hf_device_map. "
            "For this training run, that is not acceptable because it massively slows down training. "
            "Please restart the kernel, close other GPU usage, and try loading the model again."
        )
    if has_cpu_or_disk_offload:
        print(
            "WARNING: Parts of the model are offloaded to CPU or disk. "
            "This significantly slows down training. A kernel restart before the model load "
            "and as much free GPU memory as possible are then important.",
            flush=True,
        )


In [ ]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

peft_config


In [ ]:
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    max_length=resolved_max_seq_length,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    save_strategy="steps",
    save_steps=EVAL_SAVE_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    bf16=compute_dtype == torch.bfloat16,
    fp16=compute_dtype == torch.float16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=MAX_GRAD_NORM,
    completion_only_loss=True,
    packing=False,
    dataloader_num_workers=0,
    disable_tqdm=False,
    report_to="tensorboard",
    seed=SEED,
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": False,
    },
)

training_args


In [ ]:
train_examples = len(dataset["train"])
effective_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
optimizer_steps_per_epoch = math.ceil(train_examples / effective_batch_size)


def flush_print(message=""):
    print(message, flush=True)


def format_seconds(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def gpu_memory_summary():
    if not torch.cuda.is_available():
        return "cuda=n/a"
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    free_gb = free_bytes / 1024**3
    total_gb = total_bytes / 1024**3
    used_driver_gb = total_gb - free_gb
    return (
        f"gpu_alloc={allocated:.2f} GB | gpu_reserved={reserved:.2f} GB | gpu_peak={peak:.2f} GB | "
        f"cuda_free={free_gb:.2f} GB/{total_gb:.2f} GB | cuda_used_driver={used_driver_gb:.2f} GB"
    )


def summarize_batch(batch):
    parts = []
    for key, value in batch.items():
        if isinstance(value, torch.Tensor):
            parts.append(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}, device={value.device}")
    if "attention_mask" in batch and isinstance(batch["attention_mask"], torch.Tensor):
        non_pad_tokens = int(batch["attention_mask"].sum().item())
        parts.append(f"non_pad_tokens={non_pad_tokens}")
    if "labels" in batch and isinstance(batch["labels"], torch.Tensor):
        supervised_tokens = int((batch["labels"] != -100).sum().item())
        parts.append(f"supervised_tokens={supervised_tokens}")
    return " | ".join(parts)


def print_trainable_parameter_report(model, top_k=20):
    total_params = 0
    trainable_params = 0
    trainable_rows = []

    for name, param in model.named_parameters():
        param_count = param.numel()
        total_params += param_count
        if param.requires_grad:
            trainable_params += param_count
            trainable_rows.append((name, param_count, str(param.dtype), str(param.device)))

    share = 100.0 * trainable_params / max(1, total_params)
    flush_print(
        f"[trainable params] trainable={trainable_params:,} / total={total_params:,} ({share:.6f}%)"
    )

    for name, param_count, dtype, device in sorted(trainable_rows, key=lambda row: row[1], reverse=True)[:top_k]:
        flush_print(
            f"[trainable params] {name} | numel={param_count:,} | dtype={dtype} | device={device}"
        )


class ConsoleDebugCallback(TrainerCallback):
    def __init__(self, train_examples, optimizer_steps_per_epoch):
        self.train_examples = max(1, int(train_examples))
        self.optimizer_steps_per_epoch = max(1, int(optimizer_steps_per_epoch))
        self.train_start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start_time = time.time()
        flush_print(
            f"Training startet: max_steps={state.max_steps}, "
            f"train_examples_per_epoch={self.train_examples}, "
            f"optimizer_steps_per_epoch={self.optimizer_steps_per_epoch}, "
            f"geplante epochen={args.num_train_epochs}"
        )
        if DEBUG_SHOW_GPU_MEMORY:
            flush_print(f"[train begin] {gpu_memory_summary()}")
        return control

    def on_epoch_begin(self, args, state, control, **kwargs):
        flush_print(f"[epoch begin] epoch={state.epoch}")
        return control

    def on_step_begin(self, args, state, control, **kwargs):
        flush_print(f"[optimizer step begin] next_global_step={state.global_step + 1}/{state.max_steps}")
        return control

    def on_step_end(self, args, state, control, **kwargs):
        flush_print(f"[optimizer step done ] global_step={state.global_step}/{state.max_steps} | epoch={state.epoch}")
        return control

    def on_log(self, args, state, control, logs=None, **kwargs):
        elapsed = 0.0 if self.train_start_time is None else time.time() - self.train_start_time
        flush_print(f"[trainer log] elapsed={format_seconds(elapsed)} | global_step={state.global_step}/{state.max_steps} | epoch={state.epoch} | logs={logs}")
        return control

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        flush_print(f"[evaluate] epoch={state.epoch} | global_step={state.global_step}/{state.max_steps}")
        if metrics:
            flush_print(f"[evaluate] metrics={metrics}")
        return control

    def on_save(self, args, state, control, **kwargs):
        checkpoint_dir = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        flush_print(f"[save] checkpoint={checkpoint_dir}")
        return control

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = 0.0 if self.train_start_time is None else time.time() - self.train_start_time
        flush_print(f"[train end] elapsed={format_seconds(elapsed)} | best_model_checkpoint={state.best_model_checkpoint} | best_metric={state.best_metric}")
        if DEBUG_SHOW_GPU_MEMORY:
            flush_print(f"[train end] {gpu_memory_summary()}")
        return control


class DebugSFTTrainer(SFTTrainer):
    def __init__(
        self,
        *args,
        train_examples_per_epoch,
        gradient_accumulation_steps_debug,
        verbose_training=True,
        verbose_eval=True,
        show_gpu_memory=True,
        synchronize_for_timings=True,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.train_examples_per_epoch = max(1, int(train_examples_per_epoch))
        self.gradient_accumulation_steps_debug = max(1, int(gradient_accumulation_steps_debug))
        self.verbose_training = bool(verbose_training)
        self.verbose_eval = bool(verbose_eval)
        self.show_gpu_memory = bool(show_gpu_memory)
        self.synchronize_for_timings = bool(synchronize_for_timings)
        self.debug_microstep_count = 0
        self.debug_eval_batch_count = 0

    def _memory_suffix(self):
        return f" | {gpu_memory_summary()}" if self.show_gpu_memory else ""

    def training_step(self, model, inputs, num_items_in_batch=None):
        self.debug_microstep_count += 1
        epoch_index = ((self.debug_microstep_count - 1) // self.train_examples_per_epoch) + 1
        epoch_example = ((self.debug_microstep_count - 1) % self.train_examples_per_epoch) + 1
        accumulation_index = ((self.debug_microstep_count - 1) % self.gradient_accumulation_steps_debug) + 1
        approx_optimizer_step = math.ceil(self.debug_microstep_count / self.gradient_accumulation_steps_debug)

        if self.verbose_training:
            flush_print(
                f"[train microstep start] epoch={epoch_index}/{int(math.ceil(self.args.num_train_epochs))} | "
                f"example={epoch_example}/{self.train_examples_per_epoch} | "
                f"accum={accumulation_index}/{self.gradient_accumulation_steps_debug} | "
                f"approx_optimizer_step={approx_optimizer_step}/{self.state.max_steps or '?'} | "
                f"{summarize_batch(inputs)}{self._memory_suffix()}"
            )

        if torch.cuda.is_available() and self.synchronize_for_timings:
            torch.cuda.synchronize()
        start = time.time()
        loss = super().training_step(model, inputs, num_items_in_batch=num_items_in_batch)
        if torch.cuda.is_available() and self.synchronize_for_timings:
            torch.cuda.synchronize()
        elapsed = time.time() - start
        loss_value = float(loss.detach().float().cpu().item()) if isinstance(loss, torch.Tensor) else float(loss)

        if self.verbose_training:
            flush_print(
                f"[train microstep done ] epoch={epoch_index}/{int(math.ceil(self.args.num_train_epochs))} | "
                f"example={epoch_example}/{self.train_examples_per_epoch} | "
                f"accum={accumulation_index}/{self.gradient_accumulation_steps_debug} | "
                f"approx_optimizer_step={approx_optimizer_step}/{self.state.max_steps or '?'} | "
                f"step_loss={loss_value:.6f} | took={elapsed:.2f}s{self._memory_suffix()}"
            )
        return loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        self.debug_eval_batch_count += 1
        batch_index = self.debug_eval_batch_count

        if self.verbose_eval:
            flush_print(
                f"[eval batch start] batch={batch_index} | "
                f"{summarize_batch(inputs)}{self._memory_suffix()}"
            )

        if torch.cuda.is_available() and self.synchronize_for_timings:
            torch.cuda.synchronize()
        start = time.time()
        output = super().prediction_step(model, inputs, prediction_loss_only, ignore_keys=ignore_keys)
        if torch.cuda.is_available() and self.synchronize_for_timings:
            torch.cuda.synchronize()
        elapsed = time.time() - start
        loss = output[0]
        loss_text = "n/a" if loss is None else f"{float(loss.detach().float().cpu().item()):.6f}"

        if self.verbose_eval:
            flush_print(
                f"[eval batch done ] batch={batch_index} | loss={loss_text} | took={elapsed:.2f}s{self._memory_suffix()}"
            )
        return output


trainer = DebugSFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    peft_config=peft_config,
    train_examples_per_epoch=train_examples,
    gradient_accumulation_steps_debug=GRADIENT_ACCUMULATION_STEPS,
    verbose_training=DEBUG_VERBOSE_TRAINING,
    verbose_eval=DEBUG_VERBOSE_EVAL,
    show_gpu_memory=DEBUG_SHOW_GPU_MEMORY,
    synchronize_for_timings=DEBUG_SYNCHRONIZE_FOR_TIMINGS,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
        ConsoleDebugCallback(
            train_examples=train_examples,
            optimizer_steps_per_epoch=optimizer_steps_per_epoch,
        ),
    ],
)

print_trainable_parameter_report(trainer.model)

trainer


In [ ]:
def inspect_dataloader_batches(trainer, split_name, num_batches):
    if split_name == "train":
        dataloader = trainer.get_train_dataloader()
    elif split_name == "validation":
        dataloader = trainer.get_eval_dataloader()
    else:
        raise ValueError(f"Unbekannter split_name: {split_name}")

    try:
        dataloader_length = len(dataloader)
    except TypeError:
        dataloader_length = "unbekannt"

    flush_print(f"[debug dataloader] split={split_name} | type={type(dataloader).__name__} | batches={dataloader_length}")

    iterator = iter(dataloader)
    for batch_index in range(num_batches):
        try:
            batch = next(iterator)
        except StopIteration:
            flush_print(f"[debug dataloader] split={split_name} endet nach {batch_index} batches")
            break
        except Exception as exc:
            flush_print(f"[debug dataloader] split={split_name} | batch={batch_index + 1} | FEHLER beim Laden: {exc}")
            raise
        flush_print(f"[debug dataloader] split={split_name} | batch={batch_index + 1} | {summarize_batch(batch)}")


inspect_dataloader_batches(trainer, "train", DEBUG_PEEK_BATCHES)
inspect_dataloader_batches(trainer, "validation", min(DEBUG_PEEK_BATCHES, len(dataset["validation"])))

if DEBUG_RUN_BATCH_TIMING_PROBE:
    probe_batch = next(iter(trainer.get_train_dataloader()))
    prepared_probe_batch = trainer._prepare_inputs(probe_batch)

    trainer.model.train()
    trainer.model.zero_grad(set_to_none=True)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    if torch.cuda.is_available():
        probe_autocast_context = torch.autocast(device_type="cuda", dtype=compute_dtype)
    else:
        probe_autocast_context = nullcontext()

    flush_print(f"[probe] starte einen sicheren forward/loss-probe-pass auf einem einzelnen Train-Batch | {summarize_batch(prepared_probe_batch)}")
    if torch.cuda.is_available() and DEBUG_SYNCHRONIZE_FOR_TIMINGS:
        torch.cuda.synchronize()
    probe_start = time.time()
    with torch.enable_grad():
        with probe_autocast_context:
            probe_outputs = trainer.model(**prepared_probe_batch)
            probe_loss = probe_outputs.loss if hasattr(probe_outputs, "loss") else probe_outputs["loss"]
    if torch.cuda.is_available() and DEBUG_SYNCHRONIZE_FOR_TIMINGS:
        torch.cuda.synchronize()
    probe_elapsed = time.time() - probe_start
    probe_loss_value = float(probe_loss.detach().float().cpu().item())
    flush_print(f"[probe] forward/loss-pass fertig | loss={probe_loss_value:.6f} | took={probe_elapsed:.2f}s | {gpu_memory_summary()}")

    trainer.model.zero_grad(set_to_none=True)
    del probe_outputs
    del probe_loss
    trainer.debug_microstep_count = 0
    trainer.debug_eval_batch_count = 0
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    flush_print("[probe] Gradients cleared and Debug-Counter reset. After that you can start trainer.train() normally.")


In [ ]:
flush_print("[train cell] training started.")
train_wall_start = time.time()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

try:
    train_result = trainer.train()
except Exception as exc:
    flush_print(f"[train cell] trainer.train() exception: {exc}")
    if DEBUG_SHOW_GPU_MEMORY:
        flush_print(f"[train cell] GPU memory during exception: {gpu_memory_summary()}")
    raise
finally:
    train_elapsed = time.time() - train_wall_start
    flush_print(f"[train cell] trainer.train() terminated after {format_seconds(train_elapsed)}")
    if DEBUG_SHOW_GPU_MEMORY:
        flush_print(f"[train cell] GPU memory at cell end: {gpu_memory_summary()}")

trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)

train_result
